# Grupowanie w pandas i polars — agregacje, transform, lambda, tekst, pułapki

**Problem:** grupowanie to jedna z tych operacji, która w obu bibliotekach "po prostu działa" na prostych przykładach, ale ma sporo szczegółów, które ujawniają się dopiero przy realnych danych — domyślne gubienie grup z brakującym kluczem, nieprzewidywalna kolejność wyników, czy pokusa użycia `lambda` tam, gdzie jest dziesiątki razy wolniejsza od wbudowanej alternatywy.

**Porównanie:**
- pandas `groupby()` — dojrzały, bogaty w opcje, ale kilka domyślnych ustawień bywa zaskakujących (patrz Sekcja 6).
- polars `group_by()` — szybszy przy dużych danych, składnia oparta o wyrażenia (`pl.col(...)`), ale **nie gwarantuje kolejności wyników** bez jawnego parametru.

**Struktura notatnika:** od podstawowej pojedynczej agregacji, przez wielokrotne agregacje z własnymi nazwami, `transform`/`over`, `lambda`, agregacje tekstowe, aż po sekcję pułapek — każda pokazana równolegle w obu bibliotekach.

## Setup

In [ ]:
import pandas as pd
import polars as pl

data = {
    "region": ["North", "North", "South", "South", "South", "East", "East", "West"],
    "category": ["A", "B", "A", "A", "B", "A", "B", "B"],
    "salesperson": ["Anna", "Bartek", "Anna", "Celina", "Bartek", "Dawid", "Anna", "Ewa"],
    "sales": [1200, 800, 1500, 900, 1100, 700, 950, 1300],
    "units": [12, 8, 15, 9, 11, 7, 10, 13],
}

df = pd.DataFrame(data)
df_pl = pl.DataFrame(data)

df

## Sekcja 1 — Pojedyncza agregacja

In [ ]:
# pandas
df.groupby("region")["sales"].sum()

In [ ]:
# polars
df_pl.group_by("region").agg(pl.col("sales").sum())

## Sekcja 2 — Kilka agregacji naraz, z własnymi nazwami kolumn

W pandas to tzw. *named aggregation* — czytelniejsza i bezpieczniejsza forma niż stary zapis ze słownikiem (`{"sales": "sum"}`), bo pozwala nadać wynikowym kolumnom własne nazwy wprost przy definicji.

W polars każde wyrażenie w `.agg()` dostaje nazwę przez `.alias()`. Uwaga: `pl.count()` jest przestarzałe — aktualny odpowiednik to `pl.len()`.

In [ ]:
# pandas - named aggregation
df.groupby("region").agg(
    total_sales=("sales", "sum"),
    avg_sales=("sales", "mean"),
    max_units=("units", "max"),
    transactions=("sales", "count"),
)

In [ ]:
# polars
df_pl.group_by("region").agg(
    pl.col("sales").sum().alias("total_sales"),
    pl.col("sales").mean().alias("avg_sales"),
    pl.col("units").max().alias("max_units"),
    pl.len().alias("transactions"),
)

## Sekcja 3 — `transform()` (pandas) / `.over()` (polars)

W przeciwieństwie do `.agg()`, który zwraca jeden wiersz na grupę, `transform`/`over` **zwracają wynik o długości oryginalnego DataFrame** — wartość zagregowana jest "rozgłoszona" z powrotem do każdego wiersza grupy. Klasyczne zastosowanie: procent udziału wiersza w sumie jego grupy.

In [ ]:
# pandas
df["region_total_sales"] = df.groupby("region")["sales"].transform("sum")
df["pct_of_region"] = (df["sales"] / df["region_total_sales"] * 100).round(1)
df[["region", "sales", "region_total_sales", "pct_of_region"]]

In [ ]:
# polars
df_pl = df_pl.with_columns(
    pl.col("sales").sum().over("region").alias("region_total_sales")
).with_columns(
    (pl.col("sales") / pl.col("region_total_sales") * 100).round(1).alias("pct_of_region")
)
df_pl.select("region", "sales", "region_total_sales", "pct_of_region")

## Sekcja 4 — Funkcja `lambda` w agregacji

`lambda` przydaje się, gdy potrzebna logika nie ma wbudowanego odpowiednika (np. rozstęp `max - min`). W pandas to zwykły, choć wolniejszy sposób. **W polars `lambda`/`map_elements` to świadomy antywzorzec** — wymusza wykonanie wiersz-po-wierszu w Pythonie, tracąc całe przyspieszenie silnika Rust. Prawie zawsze da się to samo wyrazić natywnym wyrażeniem — patrz benchmark niżej.

In [ ]:
# pandas - lambda w .agg(), w tym w named aggregation
df.groupby("region")["sales"].agg(lambda x: x.max() - x.min())

In [ ]:
df.groupby("region").agg(sales_range=("sales", lambda x: x.max() - x.min()))

In [ ]:
# polars - ten sam wynik BEZ lambda, jako natywne wyrażenie
df_pl.group_by("region").agg(
    (pl.col("sales").max() - pl.col("sales").min()).alias("sales_range")
)

### Benchmark: `lambda`/`map_elements` vs natywne wyrażenie w polars

Na 200 000 wierszy, prosta operacja arytmetyczna na kolumnie — polars sam ostrzega o tym w runtime (`PolarsInefficientMapWarning`).

In [ ]:
import numpy as np
import time

n = 200_000
rng = np.random.default_rng(42)
big_df_pl = pl.DataFrame({
    "region": rng.choice(["North", "South", "East", "West"], n),
    "sales": rng.integers(100, 5000, n),
})

start = time.perf_counter()
big_df_pl.with_columns(
    pl.col("sales").map_elements(lambda x: x * 1.1, return_dtype=pl.Float64).alias("sales_adj")
)
t_lambda = time.perf_counter() - start

start = time.perf_counter()
big_df_pl.with_columns((pl.col("sales") * 1.1).alias("sales_adj"))
t_native = time.perf_counter() - start

print(f"map_elements (lambda, wiersz po wierszu): {t_lambda:.4f}s")
print(f"natywne wyrażenie (wektoryzowane):        {t_native:.4f}s")
print(f"Różnica: {t_lambda / t_native:.1f}x wolniej")

## Sekcja 5 — Grupowanie z tekstem

Najczęstsze potrzeby: lista unikalnych wartości tekstowych w grupie, ich liczba, pierwsza wartość, albo sklejenie w jeden string.

In [ ]:
# pandas
print(df.groupby("region")["salesperson"].nunique())
print()
print(df.groupby("region")["salesperson"].first())
print()
# Sklejenie unikalnych wartości w jeden string - wymaga lambda (brak wbudowanej metody "join")
df.groupby("region")["salesperson"].agg(lambda x: ", ".join(sorted(set(x))))

In [ ]:
# polars - te same operacje, bez lambda (wbudowane str.join)
df_pl.group_by("region").agg(
    pl.col("salesperson").n_unique().alias("num_salespeople"),
    pl.col("salesperson").first().alias("first_salesperson"),
    pl.col("salesperson").unique().sort().str.join(", ").alias("salespeople_list"),
)

## Sekcja 6 — Pułapki

### Pułapka 1 — pandas: `dropna=True` (domyślne) po cichu gubi całe grupy

Jeśli kolumna, po której grupujesz, zawiera `NaN`, domyślnie cała grupa `NaN` znika z wyniku — bez ostrzeżenia. Poniżej: suma po `groupby` wynosi 7150, mimo że suma całej kolumny to 8450 — różnica to cała utracona grupa.

In [ ]:
df_with_nan = df.copy()
df_with_nan.loc[7, "region"] = None

print(f"Suma calej kolumny 'sales':        {df_with_nan['sales'].sum()}")
print(f"Suma po groupby (dropna=True):     {df_with_nan.groupby('region')['sales'].sum().sum()}")

# Zabezpieczenie: dropna=False zachowuje grupę NaN jako osobną kategorię
df_with_nan.groupby("region", dropna=False)["sales"].sum()

### Pułapka 2 — pandas: `observed=False` na kolumnie `category` tworzy fantomowe grupy

Gdy kolumna grupująca ma typ `category` z kategoriami nieobecnymi w danych, domyślne zachowanie (`observed=False`, w praktyce już przestarzałe) tworzy w wyniku wiersz dla kategorii, która nigdy nie wystąpiła — z sumą `0`, co łatwo pomylić z rzeczywistym brakiem sprzedaży.

In [ ]:
df_cat = df.copy()
df_cat["region"] = df_cat["region"].astype("category").cat.add_categories(["Central"])

print("observed=False - 'Central' pojawia się z sumą 0, mimo braku takich danych:")
print(df_cat.groupby("region", observed=False)["sales"].sum())

print("\nobserved=True - tylko realnie występujące kategorie:")
print(df_cat.groupby("region", observed=True)["sales"].sum())

### Pułapka 3 — polars: `group_by()` nie gwarantuje kolejności wyników

W przeciwieństwie do pandas (który domyślnie sortuje po kluczu grupowania), polars domyślnie **nie gwarantuje żadnej konkretnej kolejności** wynikowych grup — z powodów wydajnościowych. Ten sam kod uruchomiony ponownie może dać inną kolejność wierszy. Jeśli kolejność ma znaczenie (np. przed eksportem albo porównaniem wyników), trzeba jawnie posortować wynik albo użyć `maintain_order=True`.

In [ ]:
print("Bez maintain_order - kolejność nieprzewidywalna:")
print(df_pl.group_by("region").agg(pl.col("sales").sum()))

print("\nZ maintain_order=True - kolejność pierwszego wystąpienia w danych źródłowych:")
print(df_pl.group_by("region", maintain_order=True).agg(pl.col("sales").sum()))

print("\nAlternatywa: jawne sortowanie wyniku (działa niezależnie od maintain_order):")
print(df_pl.group_by("region").agg(pl.col("sales").sum()).sort("region"))

### Pułapka 4 — pandas: `as_index=False` zamiast `.reset_index()`

Domyślnie `groupby()` zwraca wynik z kolumną grupującą jako indeks, nie jako zwykłą kolumnę — co potrafi zaskoczyć przy dalszym `merge()` czy zapisie do pliku. `as_index=False` załatwia to od razu, bez dodatkowego `.reset_index()` na końcu.

In [ ]:
result_default = df.groupby("region")["sales"].sum()
print(f"Domyślnie: 'region' jest indeksem -> {result_default.index.name}")

result_as_index_false = df.groupby("region", as_index=False)["sales"].sum()
result_as_index_false

## Podsumowanie

| Zadanie | pandas | polars |
|---|---|---|
| Pojedyncza agregacja | `df.groupby("col")["x"].sum()` | `df.group_by("col").agg(pl.col("x").sum())` |
| Kilka agregacji + własne nazwy | `.agg(nazwa=("x","sum"), ...)` | `.agg(pl.col("x").sum().alias("nazwa"), ...)` |
| Liczba wierszy w grupie | `("x", "count")` | `pl.len()` (nie `pl.count()` - przestarzałe) |
| Wartość zagregowana z powrotem do każdego wiersza | `.groupby("col")["x"].transform("sum")` | `pl.col("x").sum().over("col")` |
| Własna logika bez wbudowanej metody | `.agg(lambda x: ...)` (akceptowalne, ale wolniejsze) | unikaj `lambda`/`map_elements` — szukaj natywnego wyrażenia |
| Unikalne wartości tekstowe jako lista/string | `.agg(lambda x: ", ".join(sorted(set(x))))` | `pl.col("x").unique().sort().str.join(", ")` (bez lambda) |
| Grupa z brakującym kluczem | domyślnie **znika** (`dropna=True`) | `null` traktowany jako osobna grupa domyślnie |
| Kategorie nieobecne w danych (`category` dtype) | `observed=True`, żeby ich nie tworzyć fantomowo | nie dotyczy (polars nie ma typu `category` 1:1 z pandas) |
| Kolejność wyników | domyślnie sortowana po kluczu | **nieprzewidywalna** bez `maintain_order=True` lub `.sort()` |
| Kolumna grupująca jako zwykła kolumna, nie indeks | `as_index=False` | nie dotyczy — polars nigdy nie ustawia klucza jako indeksu |

**Wniosek:** największe realne ryzyko to nie brak znajomości składni, tylko domyślne ustawienia, które "po cichu" zmieniają wynik — zgubiona grupa `NaN`, fantomowa kategoria, czy nieprzewidywalna kolejność w polars. Żadne z nich nie rzuca błędu; wszystkie wymagają świadomego sprawdzenia liczby grup/wierszy przed i po, podobnie jak w notatce o `merge`/`concat`.